In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
import re
import xml.etree.ElementTree as ET
from functools import reduce
import openpyxl
from scipy.optimize import minimize_scalar
import numpy as np
import glob
import datetime as dt

pd.set_option('display.max_columns', None)

# Employment 

In [ ]:
import pandas as pd
import openpyxl

def extract_column_from_population_row(filepath, target_header, col_name):
    wb = openpyxl.load_workbook(filepath, data_only=True)
    out = []

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]

        target_col = None

        for r in range(1, min(ws.max_row, 10) + 1):
            for c in range(1, ws.max_column + 1):
                val = ws.cell(r, c).value
                if isinstance(val, str) and target_header in val:
                    target_col = c
                    break
            if target_col is not None:
                break

        if target_col is None:
            continue

        for r in range(1, ws.max_row + 1):
            label = ws.cell(r, 1).value
            value = ws.cell(r, target_col).value

            if isinstance(label, str) and ("Усе населення" in label or "Все населення" in label):
                if isinstance(value, str):
                    value = float(value.replace(",", "."))
                out.append({
                    "year": int(sheet_name),
                    col_name: value
                })
                break

    df = pd.DataFrame(out).sort_values("year").reset_index(drop=True)

    quarters = ["Q1", "Q2", "Q3", "Q4"]
    df = df.loc[df.index.repeat(4)].reset_index(drop=True)
    df["quarter"] = quarters * (len(df) // 4)
    df["year-quarter"] = df["year"].astype(str) + df["quarter"]

    df = df[["year-quarter", col_name]]

    return df

emp_16_10 = extract_column_from_population_row(
    "raw_csvs/labour_related/employment/rzn_rik16_10_u.xlsx",
    "Працездатного віку",
    'employed_percent_of_population_respective_age_group'
)

display(emp_16_10)

,year-quarter,employed_percent_of_population_respective_age_group
0,2010Q1,65.6
1,2010Q2,65.6
2,2010Q3,65.6
3,2010Q4,65.6
4,2011Q1,66.5
5,2011Q2,66.5
6,2011Q3,66.5
7,2011Q4,66.5
8,2012Q1,67.1
9,2012Q2,67.1


In [ ]:
def extract_quarter_data_from_xls(
    filepath,
    year,
    sheet_name=0,
    header_rows=(2, 3),
    data_row=8,
    start_col=1,
    end_col=9,
    quarter_col_name="year-quarter",
    people_col_name="employed_people_thousands",
    percent_col_name="employed_percent_of_population_respective_age_group"
):
    df_raw = pd.read_excel(filepath, sheet_name=sheet_name, header=None)

    header_row_1 = df_raw.iloc[header_rows[0]].tolist()
    header_row_2 = df_raw.iloc[header_rows[1]].tolist()

    combined_headers = []
    for h1, h2 in zip(header_row_1, header_row_2):
        parts = []
        for h in [h1, h2]:
            if pd.notna(h):
                s = str(h).replace("\n", " ").strip()
                if s:
                    parts.append(s)
        combined_headers.append(" | ".join(parts))

    row_values = df_raw.iloc[data_row].tolist()

    out = pd.DataFrame({
        "header": combined_headers,
        "value": row_values
    })

    out = out.iloc[start_col:end_col].reset_index(drop=True)

    if len(out) % 2 != 0:
        raise ValueError("Selected slice must contain an even number of rows.")

    n_quarters = len(out) // 2
    quarter_labels = [f"{year}Q{i}" for i in range(1, n_quarters + 1)]

    result = pd.DataFrame({
        quarter_col_name: quarter_labels,
        people_col_name: out.iloc[::2]["value"].to_list(),
        percent_col_name: out.iloc[1::2]["value"].to_list()
    })

    return result

In [ ]:
results = []
PATH_EMP = "raw_csvs/labour_related/employment/"

for year in [2017, 2018, 2019, 2020, 2021]:
    matches = list(Path(PATH_EMP).glob(f"*{year}*.xls*")) or list(Path(PATH_EMP).glob(f"*{year % 100:02d}*.xls*"))
    
    if not matches:
        print(f"No file found for {year}")
        continue
    
    if len(matches) > 1:
        print(f"Several files found for {year}: {[m.name for m in matches]}")
        file_path = matches[0]
    else:
        file_path = matches[0]
    
    df_year = extract_quarter_data_from_xls(
        filepath=file_path,
        year=year
    )
    
    results.append(df_year)

employment_quarterly = pd.concat(results, ignore_index=True)
employment_quarterly

,year-quarter,employed_people_thousands,employed_percent_of_population_respective_age_group
0,2017Q1,2424.4,73.7
1,2017Q2,2420.0,73.6
2,2017Q3,2440.1,74.2
3,2017Q4,2451.5,74.6
4,2018Q1,2457.7,74.4
5,2018Q2,2488.8,75.2
6,2018Q3,2508.9,75.8
7,2018Q4,2510.1,75.9
8,2019Q1,15570.9,66.3
9,2019Q2,15781.2,67.2


In [6]:
emp_16_10['employed_people_thousands'] = 0

employed = pd.concat([emp_16_10, employment_quarterly], ignore_index=True).reset_index(drop=True)
employed

,year-quarter,employed_percent_of_population_respective_age_group,employed_people_thousands
0,2010Q1,65.6,0.0
1,2010Q2,65.6,0.0
2,2010Q3,65.6,0.0
3,2010Q4,65.6,0.0
4,2011Q1,66.5,0.0
5,2011Q2,66.5,0.0
6,2011Q3,66.5,0.0
7,2011Q4,66.5,0.0
8,2012Q1,67.1,0.0
9,2012Q2,67.1,0.0


# Unemployment

In [7]:
unemp_16_10 = extract_column_from_population_row(
    'raw_csvs/labour_related/unemployment/bnsmv_rik16_10_u.xlsx',
    "Працездатного віку",
    'unemployed_percent_of_population_respective_age_group'
)

unemp_16_10

,year-quarter,unemployed_percent_of_population_respective_age_group
0,2010Q1,8.8
1,2010Q2,8.8
2,2010Q3,8.8
3,2010Q4,8.8
4,2011Q1,8.6
5,2011Q2,8.6
6,2011Q3,8.6
7,2011Q4,8.6
8,2012Q1,8.1
9,2012Q2,8.1


In [ ]:
results = []
PATH_UNEMP = "raw_csvs/labour_related/unemployment/"

for year in [2017, 2018, 2019, 2020, 2021]:
    matches = list(Path(PATH_UNEMP).glob(f"*{year}*.xls*")) or list(Path(PATH_UNEMP).glob(f"*{year % 100:02d}*.xls*"))
    
    if not matches:
        print(f"No file found for {year}")
        continue
    
    if len(matches) > 1:
        print(f"Several files found for {year}: {[m.name for m in matches]}")
        file_path = matches[0]
    else:
        file_path = matches[0]
    
    df_year = extract_quarter_data_from_xls(
        filepath=file_path,
        year=year,
        people_col_name="unemployed_people_thousands",
        percent_col_name="unemployed_percent_of_population_respective_age_group"
    )
    
    results.append(df_year)

unemployment_quarterly = pd.concat(results, ignore_index=True)
unemployment_quarterly

,year-quarter,unemployed_people_thousands,unemployed_percent_of_population_respective_age_group
0,2017Q1,264.6,9.8
1,2017Q2,284.8,10.5
2,2017Q3,274.5,10.1
3,2017Q4,267.2,9.8
4,2018Q1,256.8,9.5
5,2018Q2,264.7,9.6
6,2018Q3,245.4,8.9
7,2018Q4,244.6,8.9
8,2019Q1,1645.1,9.6
9,2019Q2,1527.5,8.8


In [9]:
unemp_16_10['unemployed_people_thousands'] = 0

unemployed = pd.concat([unemp_16_10, unemployment_quarterly], ignore_index=True).reset_index(drop=True)
unemployed

,year-quarter,unemployed_percent_of_population_respective_age_group,unemployed_people_thousands
0,2010Q1,8.8,0.0
1,2010Q2,8.8,0.0
2,2010Q3,8.8,0.0
3,2010Q4,8.8,0.0
4,2011Q1,8.6,0.0
5,2011Q2,8.6,0.0
6,2011Q3,8.6,0.0
7,2011Q4,8.6,0.0
8,2012Q1,8.1,0.0
9,2012Q2,8.1,0.0


# Labour force

In [10]:
lf_16_10 = extract_column_from_population_row(
    'raw_csvs/labour_related/labour_force/eansmv_rik16_10_u.xlsx',
    "Працездатного віку",
    'labour_force_percent_of_population_respective_age_group'
)

lf_16_10

,year-quarter,labour_force_percent_of_population_respective_age_group
0,2010Q1,72.0
1,2010Q2,72.0
2,2010Q3,72.0
3,2010Q4,72.0
4,2011Q1,72.7
5,2011Q2,72.7
6,2011Q3,72.7
7,2011Q4,72.7
8,2012Q1,73.0
9,2012Q2,73.0


In [ ]:
results = []
PATH_LF = "raw_csvs/labour_related/labour_force/"

for year in [2017, 2018, 2019, 2020, 2021]:
    matches = list(Path(PATH_LF).glob(f"*{year}*.xls*")) or list(Path(PATH_LF).glob(f"*{year % 100:02d}*.xls*"))
    
    if not matches:
        print(f"No file found for {year}")
        continue
    
    if len(matches) > 1:
        print(f"Several files found for {year}: {[m.name for m in matches]}")
        file_path = matches[0]
    else:
        file_path = matches[0]
    
    df_year = extract_quarter_data_from_xls(
        filepath=file_path,
        year=year,
        people_col_name="labour_force_people_thousands",
        percent_col_name="labour_force_percent_of_population_respective_age_group"
    )
    
    results.append(df_year)

labour_force_quarterly = pd.concat(results, ignore_index=True)
labour_force_quarterly

,year-quarter,labour_force_people_thousands,labour_force_percent_of_population_respective_age_group
0,2017Q1,2689.0,81.8
1,2017Q2,2704.8,82.3
2,2017Q3,2714.6,82.6
3,2017Q4,2718.7,82.7
4,2018Q1,2714.5,82.1
5,2018Q2,2753.5,83.2
6,2018Q3,2754.3,83.3
7,2018Q4,2754.7,83.3
8,2019Q1,17216.0,73.3
9,2019Q2,17308.7,73.7


In [12]:
lf_16_10['labour_force_people_thousands'] = 0

labour_force = pd.concat([lf_16_10, labour_force_quarterly], ignore_index=True).reset_index(drop=True)
labour_force

,year-quarter,labour_force_percent_of_population_respective_age_group,labour_force_people_thousands
0,2010Q1,72.0,0.0
1,2010Q2,72.0,0.0
2,2010Q3,72.0,0.0
3,2010Q4,72.0,0.0
4,2011Q1,72.7,0.0
5,2011Q2,72.7,0.0
6,2011Q3,72.7,0.0
7,2011Q4,72.7,0.0
8,2012Q1,73.0,0.0
9,2012Q2,73.0,0.0


# Unofficial employed

In [ ]:
def extract_informal_employment_annual(filepath, year, col_name="informal_employed_15_70"):
    df = pd.read_excel(filepath, sheet_name=0, header=None)

    target_row = None
    for i in range(len(df)):
        row_text = " ".join(str(x) for x in df.iloc[i].tolist() if pd.notna(x))
        if "Кількість неформально зайнятого населення" in row_text and "15-70 років" in row_text:
            target_row = i
            break

    if target_row is None:
        raise ValueError(f"Target annual row not found in {filepath}")

    target_col = None
    for i in range(min(6, len(df))):
        for j in range(df.shape[1]):
            v = df.iloc[i, j]
            if isinstance(v, str) and "Усього" in v:
                target_col = j
                break
        if target_col is not None:
            break

    if target_col is None:
        raise ValueError(f"Column 'Усього' not found in {filepath}")

    value = df.iloc[target_row, target_col]

    return pd.DataFrame({
        "year-quarter": [f"{year}Q1", f"{year}Q2", f"{year}Q3", f"{year}Q4"],
        col_name: [value, value, value, value]
    })


def extract_informal_employment_quarterly(filepath, year, col_name="informal_employed_15_70"):
    df = pd.read_excel(filepath, sheet_name=0, header=None)

    target_row = None
    for i in range(len(df)):
        row_text = " ".join(str(x) for x in df.iloc[i].tolist() if pd.notna(x))
        if "з нього 15-70 років" in row_text:
            target_row = i
            break

    if target_row is None:
        raise ValueError(f"Target quarterly row not found in {filepath}")

    values = pd.to_numeric(df.iloc[target_row, 1:5], errors="coerce").tolist()

    return pd.DataFrame({
        "year-quarter": [f"{year}Q1", f"{year}Q2", f"{year}Q3", f"{year}Q4"],
        col_name: values
    })


def extract_informal_employment_for_years(path_folder, years_annual, years_quarterly,
                                          col_name="informal_employed_15_70"):
    all_results = []

    for year in years_annual:
        matches = list(Path(path_folder).glob(f"*{year}*.xls*"))
        if not matches:
            matches = list(Path(path_folder).glob(f"*{year % 100:02d}*.xls*"))

        if not matches:
            print(f"No annual file found for {year}")
            continue

        matches = [m for m in matches if "smpvg" in m.name.lower()]
        if not matches:
            print(f"No suitable annual file found for {year}")
            continue

        file_path = matches[0]
        df_year = extract_informal_employment_annual(file_path, year, col_name=col_name)
        all_results.append(df_year)

    for year in years_quarterly:
        matches = list(Path(path_folder).glob(f"*{year}*.xls*"))
        if not matches:
            matches = list(Path(path_folder).glob(f"*{year % 100:02d}*.xls*"))

        if not matches:
            print(f"No quarterly file found for {year}")
            continue

        matches = [m for m in matches if "tmvg" in m.name.lower() or "stmvg" in m.name.lower()]
        if not matches:
            print(f"No suitable quarterly file found for {year}")
            continue

        file_path = matches[0]
        df_year = extract_informal_employment_quarterly(file_path, year, col_name=col_name)
        all_results.append(df_year)

    result = pd.concat(all_results, ignore_index=True)
    return result.sort_values("year-quarter").reset_index(drop=True)

In [14]:
PATH_EMP = "raw_csvs/labour_related/self_employment"

informal_emp = extract_informal_employment_for_years(
    path_folder=PATH_EMP,
    years_annual=[2017, 2018, 2019],
    years_quarterly=[2020, 2021],
    col_name="informal_employed_15_70_thousands"
)

display(informal_emp)

,year-quarter,informal_employed_15_70_thousands
0,2017Q1,3695.6
1,2017Q2,3695.6
2,2017Q3,3695.6
3,2017Q4,3695.6
4,2018Q1,3541.3
5,2018Q2,3541.3
6,2018Q3,3541.3
7,2018Q4,3541.3
8,2019Q1,3460.4
9,2019Q2,3460.4


# Population

In [16]:
url = "https://index.minfin.com.ua/ua/reference/people/"
html = requests.get(url, timeout=30).text

soup = BeautifulSoup(html, "html.parser")
text = soup.get_text("\n", strip=True).replace("\xa0", " ").replace("\u202f", " ")

pattern = re.compile(r"(на\s+1\.01\.(19\d{2}|20[0-1]\d|2020|2021|2022))\s+([\d\s]+,\d+)")
matches = pattern.findall(text)

df = pd.DataFrame(
    [(m[0], m[2]) for m in matches],
    columns=["Дата", "Чисельність (тис.)"]
)

df["Чисельність (тис.)"] = (
    df["Чисельність (тис.)"]
    .str.replace(" ", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

df['Чисельність (тис.)'] = df['Чисельність (тис.)'] * 1000

df["Дата"] = df["Дата"].str.replace("на\n", "", regex=False)
df["Дата"] = pd.to_datetime(df["Дата"], format="%d.%m.%Y")

df = (
    df.drop_duplicates(subset="Дата")
      .sort_values("Дата")
      .reset_index(drop=True)
)

df.columns = ["Date", "Population"]

df = df[df["Date"] >= dt.datetime(2010, 1, 1)]

df

,Date,Population
20,2010-01-01,45962900.0
21,2011-01-01,45778500.0
22,2012-01-01,45633600.0
23,2013-01-01,45553000.0
24,2014-01-01,45426200.0
25,2015-01-01,42928900.0
26,2016-01-01,42760500.0
27,2017-01-01,42584500.0
28,2018-01-01,42386400.0
29,2019-01-01,42153200.0


In [ ]:
df_q = df.copy()
df_q["Date"] = pd.to_datetime(df_q["Date"])
df_q["year"] = df_q["Date"].dt.year

df_q = df_q.loc[df_q.index.repeat(4)].reset_index(drop=True)
df_q["quarter"] = ["Q1", "Q2", "Q3", "Q4"] * (len(df_q) // 4)
df_q["year_quarter"] = df_q["year"].astype(str) + df_q["quarter"]

df_q = df_q[["year_quarter", "Population"]]
df_q.head()

,year_quarter,Population
0,2010Q1,45962900.0
1,2010Q2,45962900.0
2,2010Q3,45962900.0
3,2010Q4,45962900.0
4,2011Q1,45778500.0


# Final

In [18]:
final = employed.merge(unemployed, on="year-quarter", how="outer") \
    .merge(labour_force, on="year-quarter", how="outer") \
    .merge(informal_emp, on="year-quarter", how="outer") \

final = final.rename(columns={"year-quarter": "year_quarter"}).merge(df_q, on="year_quarter", how="outer")
final.to_csv("data/final_labour_data.csv", index=False)

In [19]:
final

,year_quarter,employed_percent_of_population_respective_age_group,employed_people_thousands,unemployed_percent_of_population_respective_age_group,unemployed_people_thousands,labour_force_percent_of_population_respective_age_group,labour_force_people_thousands,informal_employed_15_70_thousands,Population
0,2010Q1,65.6,0.0,8.8,0.0,72.0,0.0,NaN,45962900.0
1,2010Q2,65.6,0.0,8.8,0.0,72.0,0.0,NaN,45962900.0
2,2010Q3,65.6,0.0,8.8,0.0,72.0,0.0,NaN,45962900.0
3,2010Q4,65.6,0.0,8.8,0.0,72.0,0.0,NaN,45962900.0
4,2011Q1,66.5,0.0,8.6,0.0,72.7,0.0,NaN,45778500.0
5,2011Q2,66.5,0.0,8.6,0.0,72.7,0.0,NaN,45778500.0
6,2011Q3,66.5,0.0,8.6,0.0,72.7,0.0,NaN,45778500.0
7,2011Q4,66.5,0.0,8.6,0.0,72.7,0.0,NaN,45778500.0
8,2012Q1,67.1,0.0,8.1,0.0,73.0,0.0,NaN,45633600.0
9,2012Q2,67.1,0.0,8.1,0.0,73.0,0.0,NaN,45633600.0
